# Decision fusion, focal-loss models (comparison)

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 3.4, 4.4.

Same fusion analysis for the focal-loss CNNs, including the Dempster–Shafer rules. Comparison only; not the final system.

**Environment.** Written for Google Colab (CPU is enough); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** Focal-loss CNN and DistilBERT predictions on UrbanSound8K.

**Outputs.** Fusion metrics for every rule and model pair.

**Note.** Uses the focal-loss zone 0.35–0.60. The final results reported in the paper are produced by `final_fusion_evaluation.py`.


# UrbanSound Focal-Loss-Trained CNN + DistilBERT Fusion Rules

This notebook combines the three focal-loss-trained CNN fusion notebooks into one all-rules evaluation.

It evaluates 12 fusion rules for every CNN x DistilBERT pair:

- `cnn_only`
- `distilbert_only`
- `override`
- `weighted_average`
- `cascade_correction`
- `probabilistic_late`
- `custom_uncertain_handoff`
- `cascade_near_boundary_rescue`
- `ds_fusion`
- `ds_fusion_no_theta`
- `or_rule`
- `and_rule`

`ds_fusion` combines CNN and DistilBERT danger probabilities using Dempster-Shafer evidence theory with fixed uncertainty masses of 0.10 for CNN and 0.05 for DistilBERT.

The `and_rule` can output `unknown` when CNN and DistilBERT disagree. Its binary metrics are computed only on covered rows, and the metrics table records `unknown_rows` and `coverage_rate`.

It exports one metrics file containing all pairs and all rules, plus diagnostics, pair matrices, best-rule summaries, and plots.


## 1. Mount Drive and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 80)


## 2. Paths and Model Lists

Edit these paths if your Drive folders move. The defaults match the original focal-loss fusion notebooks.


In [ ]:
DISTILBERT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/DistilBERT/training/low_threshold_experiments/external_eval_urbansound')
CNN_EPOCH_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/FIRST TRY/CNNs/UrbanSound_test')
CNN_SELECTED_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/CNNs/UrbanSound_test')

OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/FusionRules/FINAL')
PLOT_DIR = OUTPUT_DIR / 'plots'
PAIR_DIR = OUTPUT_DIR / 'pair_matrices'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PAIR_DIR.mkdir(parents=True, exist_ok=True)

CNN_EPOCH_DIRS = [
    'DangerDetective_1DWAVE_LV-81_s8_p1__best_epoch0850',
    'DangerDetective_FeatureExtract__best_epoch0100',
    'DangerDetective_1DWAVE_LV-18_s1_p8__best_epoch0750',
    'DangerDetective_1DWAVE_LV-24_s2_p4__best_epoch0300',
    'DangerDetective_1DWAVE_LV-42_s4_p2__best_epoch0100',
]

CNN_SELECTED_DIRS = [
    'DangerDetective_1DWAVE_LV-42_s4_p2__selected_by_val_loss',
    'DangerDetective_1DWAVE_LV-81_s8_p1__selected_by_val_auc_roc',
    'DangerDetective_1DWAVE_LV-81_s8_p1__selected_by_val_loss',
    'DangerDetective_FeatureExtract__selected_by_val_auc_roc',
    'DangerDetective_FeatureExtract__selected_by_val_loss',
    'DangerDetective_1DWAVE_LV-18_s1_p8__selected_by_val_auc_roc',
    'DangerDetective_1DWAVE_LV-18_s1_p8__selected_by_val_loss',
    'DangerDetective_1DWAVE_LV-24_s2_p4__selected_by_val_auc_roc',
    'DangerDetective_1DWAVE_LV-24_s2_p4__selected_by_val_loss',
    'DangerDetective_1DWAVE_LV-42_s4_p2__selected_by_val_auc_roc',
]

CNN_MODEL_SPECS = [
    {'root': CNN_EPOCH_ROOT, 'name': name, 'selection_group': 'epoch_based'}
    for name in CNN_EPOCH_DIRS
] + [
    {'root': CNN_SELECTED_ROOT, 'name': name, 'selection_group': 'selected'}
    for name in CNN_SELECTED_DIRS
]

DISTILBERT_DIRS = [
    'low_threshold__random__focal_loss',
    'low_threshold__random__standard_ce',
    'low_threshold__random__weighted_ce',
    'low_threshold__alignment_stratified__focal_loss',
    'low_threshold__alignment_stratified__standard_ce',
    'low_threshold__alignment_stratified__weighted_ce',
]

print('DISTILBERT_ROOT =', DISTILBERT_ROOT)
print('CNN_EPOCH_ROOT =', CNN_EPOCH_ROOT)
print('CNN_SELECTED_ROOT =', CNN_SELECTED_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('CNN model specs:', len(CNN_MODEL_SPECS))
print('DistilBERT models:', len(DISTILBERT_DIRS))


## 3. Fusion Rule Settings

In [ ]:
CNN_THRESHOLD = 0.50
FUSION_THRESHOLD = 0.50
CASCADE_THRESHOLD = 0.35
UNCERTAIN_LOW = 0.35
UNCERTAIN_HIGH = 0.60
MIDRANGE_CASCADE_LOW = 0.35
MIDRANGE_CASCADE_HIGH = 0.50
CNN_WEIGHT = 0.50
DIST_WEIGHT = 0.50
UNKNOWN_PRED = -1
DS_UNCERTAINTY_CNN = 0.10
DS_UNCERTAINTY_DISTILBERT = 0.05

RULE_DESCRIPTIONS = pd.DataFrame([
    {'rule': 'cnn_only', 'meaning': 'Use CNN probability only.'},
    {'rule': 'distilbert_only', 'meaning': 'Use DistilBERT probability only.'},
    {'rule': 'override', 'meaning': 'If DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability.'},
    {'rule': 'weighted_average', 'meaning': 'Average CNN and DistilBERT probabilities with equal weights.'},
    {'rule': 'cascade_correction', 'meaning': 'If CNN probability < 0.35 and DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability.'},
    {'rule': 'probabilistic_late', 'meaning': 'If CNN probability < 0.35, use weighted average probability; otherwise keep CNN probability.'},
    {'rule': 'custom_uncertain_handoff', 'meaning': 'If 0.35 <= CNN probability <= 0.60, use DistilBERT probability; otherwise keep CNN probability.'},
    {'rule': 'cascade_near_boundary_rescue', 'meaning': 'If 0.35 <= CNN probability <= 0.50 and DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability.'},
    {'rule': 'ds_fusion', 'meaning': 'Dempster-Shafer late fusion using uncertainty masses 0.10 for CNN and 0.05 for DistilBERT.'},
    {'rule': 'ds_fusion_no_theta', 'meaning': 'Dempster-Shafer-style fusion over danger and safe only, with no uncertainty/theta mass.'},
    {'rule': 'or_rule', 'meaning': 'If either CNN or DistilBERT predicts danger, output danger; if both predict no danger, output no danger.'},
    {'rule': 'and_rule', 'meaning': 'If both models agree, output their shared class; if they disagree, output unknown.'},
])

RULE_DESCRIPTIONS


## 4. Helper Functions

In [ ]:
def slug(text: str) -> str:
    text = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_')
    return re.sub(r'_+', '_', text).lower()


def model_short_name(name: str) -> str:
    return (
        name.replace('DangerDetective_', '')
        .replace('low_threshold__', '')
        .replace('__selected_by_', '__sel_')
        .replace('__best_epoch', '__best')
        .replace('alignment_stratified', 'align')
    )


def normalize_label(values: pd.Series) -> pd.Series:
    raw = values.astype(str).str.strip().str.lower()
    return raw.replace({
        '0': 'no_danger_risk',
        '1': 'danger_risk',
        'no danger risk': 'no_danger_risk',
        'danger risk': 'danger_risk',
        'not_danger_related': 'no_danger_risk',
        'danger_related': 'danger_risk',
    })


def compute_metrics(y_true, prob, pred) -> dict:
    y_true_all = pd.to_numeric(pd.Series(y_true), errors='coerce').astype(int)
    prob_all = pd.to_numeric(pd.Series(prob), errors='coerce')
    pred_all = pd.to_numeric(pd.Series(pred), errors='coerce').astype(int)

    known_mask = pred_all.isin([0, 1])
    y_true = y_true_all[known_mask]
    prob = prob_all[known_mask]
    pred = pred_all[known_mask]

    result = {
        'total_rows': int(len(pred_all)),
        'evaluated_rows': int(known_mask.sum()),
        'unknown_rows': int((~known_mask).sum()),
        'coverage_rate': float(known_mask.mean()) if len(known_mask) else np.nan,
    }

    if len(pred) == 0:
        result.update({
            'accuracy': np.nan,
            'macro_f1': np.nan,
            'precision_no_danger': np.nan,
            'recall_no_danger': np.nan,
            'f1_no_danger': np.nan,
            'support_no_danger': 0,
            'precision_danger': np.nan,
            'recall_danger': np.nan,
            'f1_danger': np.nan,
            'support_danger': 0,
            'tn': 0,
            'fp': 0,
            'fn': 0,
            'tp': 0,
            'false_positive_rate': np.nan,
            'false_negative_rate': np.nan,
            'roc_auc': np.nan,
            'average_precision': np.nan,
        })
        return result

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, pred, labels=[0, 1], zero_division=0
    )

    result.update({
        'accuracy': accuracy_score(y_true, pred),
        'macro_f1': float((f1[0] + f1[1]) / 2),
        'precision_no_danger': precision[0],
        'recall_no_danger': recall[0],
        'f1_no_danger': f1[0],
        'support_no_danger': int(support[0]),
        'precision_danger': precision[1],
        'recall_danger': recall[1],
        'f1_danger': f1[1],
        'support_danger': int(support[1]),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'false_positive_rate': int(cm[0, 1]) / max(int(cm[0, 0] + cm[0, 1]), 1),
        'false_negative_rate': int(cm[1, 0]) / max(int(cm[1, 0] + cm[1, 1]), 1),
    })
    if y_true.nunique() == 2 and prob.notna().all():
        result['roc_auc'] = roc_auc_score(y_true, prob)
        result['average_precision'] = average_precision_score(y_true, prob)
    else:
        result['roc_auc'] = np.nan
        result['average_precision'] = np.nan
    return result


## 5. Load and Standardize Predictions

In [ ]:
def load_cnn_model(model_spec: dict) -> pd.DataFrame:
    model_dir_name = model_spec['name']
    root = model_spec['root']
    selection_group = model_spec.get('selection_group', '')
    path = root / model_dir_name / 'urbansound_unknown_test_predictions_detailed.csv'
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    out = pd.DataFrame({
        'filename': df['filename'].astype(str).map(lambda x: Path(x).name),
        'caption': df.get('caption', '').astype(str),
        'class': df.get('class', df.get('category', '')).astype(str),
        'y_true': pd.to_numeric(df['y_true'], errors='coerce').astype(int),
        'reference_label': normalize_label(df['semantic_label_final']),
        'model_family': 'CNN',
        'model_name': model_dir_name,
        'model_short': model_short_name(model_dir_name),
        'selection_group': selection_group,
        'prob_danger': pd.to_numeric(df['prob_danger_risk'], errors='coerce'),
    })
    out['prediction'] = (out['prob_danger'] >= CNN_THRESHOLD).astype(int)
    out['prediction_label'] = out['prediction'].map({0: 'no_danger_risk', 1: 'danger_risk'})
    return out


def load_distilbert_model(model_dir_name: str) -> pd.DataFrame:
    path = DISTILBERT_ROOT / model_dir_name / 'urbansound8k_predictions.csv'
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    out = pd.DataFrame({
        'filename': df['filename'].astype(str).map(lambda x: Path(x).name),
        'caption': df.get('caption', '').astype(str),
        'class': df.get('class', df.get('original_class_for_eval', '')).astype(str),
        'y_true': pd.to_numeric(df['y_true'], errors='coerce').astype(int),
        'reference_label': normalize_label(df['semantic_label_final']),
        'model_family': 'DistilBERT',
        'model_name': model_dir_name,
        'model_short': model_short_name(model_dir_name),
        'selection_group': 'distilbert',
        'prob_danger': pd.to_numeric(df['distilbert_danger_prob'], errors='coerce'),
    })
    out['prediction'] = (out['prob_danger'] >= FUSION_THRESHOLD).astype(int)
    out['prediction_label'] = out['prediction'].map({0: 'no_danger_risk', 1: 'danger_risk'})
    return out


cnn_df = pd.concat([load_cnn_model(spec) for spec in CNN_MODEL_SPECS], ignore_index=True)
dist_df = pd.concat([load_distilbert_model(name) for name in DISTILBERT_DIRS], ignore_index=True)
prob_long = pd.concat([cnn_df, dist_df], ignore_index=True)

print('CNN models:', cnn_df['model_name'].nunique(), 'rows:', len(cnn_df))
print('DistilBERT models:', dist_df['model_name'].nunique(), 'rows:', len(dist_df))
print('Unique files in CNN:', cnn_df['filename'].nunique())
print('Unique files in DistilBERT:', dist_df['filename'].nunique())

prob_long.to_csv(OUTPUT_DIR / 'all_model_probability_long.csv', index=False)
prob_long.head()


## 6. Fusion Rule Implementation

In [ ]:
RULE_PROB_COLS = [
    'rule_cnn_only_prob',
    'rule_distilbert_only_prob',
    'rule_override_prob',
    'rule_weighted_average_prob',
    'rule_cascade_correction_prob',
    'rule_probabilistic_late_prob',
    'rule_custom_uncertain_handoff_prob',
    'rule_cascade_near_boundary_rescue_prob',
    'rule_ds_fusion_prob',
    'rule_ds_fusion_no_theta_prob',
    'rule_or_rule_prob',
    'rule_and_rule_prob',
]


def fusion_probabilities(pair_df: pd.DataFrame) -> pd.DataFrame:
    out = pair_df.copy()
    cnn = out['cnn_prob_danger']
    dist = out['dist_prob_danger']
    cnn_pred = out['cnn_pred'].astype(int)
    dist_pred = out['dist_pred'].astype(int)
    weighted = ((CNN_WEIGHT * cnn) + (DIST_WEIGHT * dist)) / (CNN_WEIGHT + DIST_WEIGHT)

    cascade = cnn < CASCADE_THRESHOLD
    uncertain = (cnn >= UNCERTAIN_LOW) & (cnn <= UNCERTAIN_HIGH)
    near_boundary = (cnn >= MIDRANGE_CASCADE_LOW) & (cnn <= MIDRANGE_CASCADE_HIGH)
    models_agree = cnn_pred.eq(dist_pred)

    out['rule_cnn_only_prob'] = cnn
    out['rule_distilbert_only_prob'] = dist
    out['rule_override_prob'] = np.where(dist_pred == 1, dist, cnn)
    out['rule_weighted_average_prob'] = weighted
    out['rule_cascade_correction_prob'] = np.where(cascade & (dist_pred == 1), dist, cnn)
    out['rule_probabilistic_late_prob'] = np.where(cascade, weighted, cnn)
    out['rule_custom_uncertain_handoff_prob'] = np.where(uncertain, dist, cnn)
    out['rule_cascade_near_boundary_rescue_prob'] = np.where(near_boundary & (dist_pred == 1), dist, cnn)

    m1_danger = cnn * (1 - DS_UNCERTAINTY_CNN)
    m1_safe = (1 - cnn) * (1 - DS_UNCERTAINTY_CNN)
    m1_theta = DS_UNCERTAINTY_CNN
    m2_danger = dist * (1 - DS_UNCERTAINTY_DISTILBERT)
    m2_safe = (1 - dist) * (1 - DS_UNCERTAINTY_DISTILBERT)
    m2_theta = DS_UNCERTAINTY_DISTILBERT
    conflict = (m1_danger * m2_safe) + (m1_safe * m2_danger)
    k = 1 - conflict

    fused_danger = (
        (m1_danger * m2_danger) + (m1_danger * m2_theta) + (m1_theta * m2_danger)
    ) / k
    fused_safe = (
        (m1_safe * m2_safe) + (m1_safe * m2_theta) + (m1_theta * m2_safe)
    ) / k

    out['rule_ds_fusion_prob'] = fused_danger
    out['rule_ds_fusion_pred'] = np.where(fused_danger > fused_safe, 1, 0).astype(int)

    no_theta_conflict = (cnn * (1 - dist)) + ((1 - cnn) * dist)
    no_theta_k = np.maximum(1 - no_theta_conflict, 1e-12)
    no_theta_danger = (cnn * dist) / no_theta_k
    no_theta_safe = ((1 - cnn) * (1 - dist)) / no_theta_k
    out['rule_ds_fusion_no_theta_prob'] = no_theta_danger
    out['rule_ds_fusion_no_theta_pred'] = np.where(
        no_theta_danger > no_theta_safe, 1, 0
    ).astype(int)

    out['rule_or_rule_pred'] = np.where((cnn_pred == 1) | (dist_pred == 1), 1, 0).astype(int)
    out['rule_or_rule_prob'] = np.maximum(cnn, dist)

    out['rule_and_rule_pred'] = np.where(models_agree, cnn_pred, UNKNOWN_PRED).astype(int)
    out['rule_and_rule_prob'] = np.where(
        models_agree,
        np.where(cnn_pred == 1, np.minimum(cnn, dist), np.maximum(cnn, dist)),
        np.nan,
    )
    out['rule_and_rule_label'] = pd.Series(out['rule_and_rule_pred']).map({
        0: 'no_danger_risk',
        1: 'danger_risk',
        UNKNOWN_PRED: 'unknown',
    })

    out['cascade_triggered'] = cascade
    out['custom_uncertain_triggered'] = uncertain
    out['near_boundary_rescue_triggered'] = near_boundary
    out['near_boundary_rescue_used_distilbert'] = near_boundary & (dist_pred == 1)
    out['or_rule_triggered'] = cnn_pred.ne(dist_pred)
    out['and_rule_unknown'] = out['rule_and_rule_pred'].eq(UNKNOWN_PRED)

    for col in RULE_PROB_COLS:
        pred_col = col.replace('_prob', '_pred')
        if pred_col not in out.columns:
            out[pred_col] = (out[col] >= FUSION_THRESHOLD).astype(int)
    return out


def build_pair_matrix(cnn_model: pd.DataFrame, dist_model: pd.DataFrame) -> pd.DataFrame:
    cnn_cols = ['filename', 'caption', 'class', 'y_true', 'reference_label', 'prob_danger', 'prediction']
    dist_cols = ['filename', 'prob_danger', 'prediction']
    merged = cnn_model[cnn_cols].merge(
        dist_model[dist_cols],
        on='filename',
        how='inner',
        suffixes=('_cnn', '_dist'),
    )
    merged = merged.rename(columns={
        'prob_danger_cnn': 'cnn_prob_danger',
        'prob_danger_dist': 'dist_prob_danger',
        'prediction_cnn': 'cnn_pred',
        'prediction_dist': 'dist_pred',
    })
    return fusion_probabilities(merged)


## 7. Evaluate Every CNN x DistilBERT Pair

In [ ]:
metric_rows = []
diagnostic_rows = []
matrix_frames = []

for cnn_name, cnn_model in cnn_df.groupby('model_name'):
    for dist_name, dist_model in dist_df.groupby('model_name'):
        pair = build_pair_matrix(cnn_model, dist_model)
        cnn_short = model_short_name(cnn_name)
        dist_short = model_short_name(dist_name)
        pair_name = f'{cnn_short}__X__{dist_short}'
        pair.insert(0, 'cnn_model', cnn_name)
        pair.insert(1, 'distilbert_model', dist_name)
        pair.insert(2, 'pair_name', pair_name)

        matrix_cols = [
            'pair_name', 'cnn_model', 'distilbert_model', 'filename', 'class', 'caption',
            'y_true', 'reference_label', 'cnn_prob_danger', 'dist_prob_danger',
            *RULE_PROB_COLS,
            'cnn_pred', 'dist_pred',
            *[c.replace('_prob', '_pred') for c in RULE_PROB_COLS],
            'rule_and_rule_label',
            'cascade_triggered', 'custom_uncertain_triggered',
            'near_boundary_rescue_triggered', 'near_boundary_rescue_used_distilbert',
            'or_rule_triggered', 'and_rule_unknown',
        ]
        pair[matrix_cols].to_csv(PAIR_DIR / f'{slug(pair_name)}_all_rules_probability_matrix.csv', index=False)
        matrix_frames.append(pair[matrix_cols])

        for rule_col in RULE_PROB_COLS:
            rule_name = rule_col.replace('rule_', '').replace('_prob', '')
            pred_col = rule_col.replace('_prob', '_pred')
            metrics = compute_metrics(pair['y_true'], pair[rule_col], pair[pred_col])

            cnn_wrong = pair['cnn_pred'].ne(pair['y_true'])
            fusion_unknown = pair[pred_col].eq(UNKNOWN_PRED)
            fusion_correct = pair[pred_col].eq(pair['y_true'])

            metrics.update({
                'cnn_model': cnn_name,
                'distilbert_model': dist_name,
                'pair_name': pair_name,
                'rule_name': rule_name,
                'n_rows': len(pair),
                'cnn_misclassified_rows': int(cnn_wrong.sum()),
                'cnn_misclassified_fixed_by_fusion_rows': int((cnn_wrong & fusion_correct).sum()),
                'cnn_misclassified_not_fixed_by_fusion_rows': int((cnn_wrong & ~fusion_correct).sum()),
                'cnn_misclassified_unknown_by_fusion_rows': int((cnn_wrong & fusion_unknown).sum()),
                'cnn_correct_hurt_by_fusion_rows': int((~cnn_wrong & ~fusion_correct & ~fusion_unknown).sum()),
            })

            if rule_name == 'cascade_correction':
                trigger = pair['cascade_triggered']
                used = trigger & (pair['dist_pred'] == 1)
            elif rule_name == 'probabilistic_late':
                trigger = pair['cascade_triggered']
                used = trigger
            elif rule_name == 'custom_uncertain_handoff':
                trigger = pair['custom_uncertain_triggered']
                used = trigger
            elif rule_name == 'cascade_near_boundary_rescue':
                trigger = pair['near_boundary_rescue_triggered']
                used = pair['near_boundary_rescue_used_distilbert']
            elif rule_name in ['ds_fusion', 'ds_fusion_no_theta']:
                trigger = pd.Series(True, index=pair.index)
                used = trigger
            elif rule_name == 'or_rule':
                trigger = pair['or_rule_triggered']
                used = trigger
            elif rule_name == 'and_rule':
                trigger = pair['and_rule_unknown']
                used = ~pair['and_rule_unknown']
            else:
                trigger = pd.Series(False, index=pair.index)
                used = pd.Series(False, index=pair.index)

            metrics['triggered_rows'] = int(trigger.sum())
            metrics['used_distilbert_rows'] = int(used.sum())
            metric_rows.append(metrics)

            if trigger.any():
                triggered = pair[trigger].copy()
                cnn_correct = triggered['cnn_pred'].eq(triggered['y_true'])
                dist_correct = triggered['dist_pred'].eq(triggered['y_true'])
                triggered_fusion_unknown = triggered[pred_col].eq(UNKNOWN_PRED)
                triggered_fusion_correct = triggered[pred_col].eq(triggered['y_true'])
                diagnostic_rows.append({
                    'cnn_model': cnn_name,
                    'distilbert_model': dist_name,
                    'pair_name': pair_name,
                    'rule_name': rule_name,
                    'triggered_rows': int(trigger.sum()),
                    'used_distilbert_rows': int(used.sum()),
                    'unknown_rows_triggered': int(triggered_fusion_unknown.sum()),
                    'triggered_rate': float(trigger.mean()),
                    'used_distilbert_rate': float(used.mean()),
                    'cnn_accuracy_triggered': float(cnn_correct.mean()),
                    'distilbert_accuracy_triggered': float(dist_correct.mean()),
                    'fusion_accuracy_triggered': float(triggered_fusion_correct.mean()),
                    'fusion_helped_count': int((~cnn_correct & triggered_fusion_correct).sum()),
                    'fusion_hurt_count': int((cnn_correct & ~triggered_fusion_correct & ~triggered_fusion_unknown).sum()),
                    'fusion_unknown_on_cnn_wrong_count': int((~cnn_correct & triggered_fusion_unknown).sum()),
                    'fusion_net_help': int((~cnn_correct & triggered_fusion_correct).sum() - (cnn_correct & ~triggered_fusion_correct & ~triggered_fusion_unknown).sum()),
                    'changed_prediction_count': int(triggered['cnn_pred'].ne(triggered[pred_col]).sum()),
                })

all_metrics = pd.DataFrame(metric_rows).sort_values(
    ['macro_f1', 'accuracy', 'recall_danger'], ascending=False
)
all_diagnostics = pd.DataFrame(diagnostic_rows).sort_values(
    ['fusion_net_help', 'fusion_accuracy_triggered'], ascending=False
)
all_pair_matrix_long = pd.concat(matrix_frames, ignore_index=True)

all_metrics.to_csv(OUTPUT_DIR / 'fusion_metrics_all_pairs_all_rules.csv', index=False)
all_diagnostics.to_csv(OUTPUT_DIR / 'fusion_rule_diagnostics.csv', index=False)
all_pair_matrix_long.to_csv(OUTPUT_DIR / 'fusion_probability_matrix_long.csv', index=False)

print('Pair matrices saved to:', PAIR_DIR)
print('Metrics saved to:', OUTPUT_DIR / 'fusion_metrics_all_pairs_all_rules.csv')
print('Diagnostics saved to:', OUTPUT_DIR / 'fusion_rule_diagnostics.csv')
print('Long matrix saved to:', OUTPUT_DIR / 'fusion_probability_matrix_long.csv')

all_metrics.head(30)


## 8. Best Rule Summaries

In [ ]:
best_rule_per_pair = (
    all_metrics
    .sort_values(['pair_name', 'macro_f1', 'accuracy', 'recall_danger'], ascending=[True, False, False, False])
    .groupby('pair_name', as_index=False)
    .head(1)
    .sort_values(['macro_f1', 'accuracy', 'recall_danger'], ascending=False)
)

best_pair_per_rule = (
    all_metrics
    .sort_values(['rule_name', 'macro_f1', 'accuracy', 'recall_danger'], ascending=[True, False, False, False])
    .groupby('rule_name', as_index=False)
    .head(1)
    .sort_values(['macro_f1', 'accuracy', 'recall_danger'], ascending=False)
)

rule_summary = (
    all_metrics
    .groupby('rule_name')
    .agg(
        mean_macro_f1=('macro_f1', 'mean'),
        max_macro_f1=('macro_f1', 'max'),
        mean_accuracy=('accuracy', 'mean'),
        mean_recall_danger=('recall_danger', 'mean'),
        mean_f1_danger=('f1_danger', 'mean'),
        mean_false_positive_rate=('false_positive_rate', 'mean'),
        mean_false_negative_rate=('false_negative_rate', 'mean'),
        mean_triggered_rows=('triggered_rows', 'mean'),
        mean_used_distilbert_rows=('used_distilbert_rows', 'mean'),
    )
    .reset_index()
    .sort_values(['max_macro_f1', 'mean_macro_f1'], ascending=False)
)

best_rule_per_pair.to_csv(OUTPUT_DIR / 'best_rule_per_pair_with_near_boundary_and_borderline_rescue.csv', index=False)
best_pair_per_rule.to_csv(OUTPUT_DIR / 'best_pair_per_rule_with_near_boundary_and_borderline_rescue.csv', index=False)
rule_summary.to_csv(OUTPUT_DIR / 'rule_summary_with_near_boundary_and_borderline_rescue.csv', index=False)

print('Best rule per pair saved to:', OUTPUT_DIR / 'best_rule_per_pair_with_near_boundary_and_borderline_rescue.csv')
print('Best pair per rule saved to:', OUTPUT_DIR / 'best_pair_per_rule_with_near_boundary_and_borderline_rescue.csv')
print('Rule summary saved to:', OUTPUT_DIR / 'rule_summary_with_near_boundary_and_borderline_rescue.csv')

rule_summary


## 9. Optional Quick Plots

In [ ]:
plt.figure(figsize=(13, 5))
all_metrics.boxplot(column='macro_f1', by='rule_name', rot=45)
plt.title('Macro F1 by Fusion Rule')
plt.suptitle('')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'macro_f1_by_rule_boxplot.png', dpi=200)
plt.show()

plt.figure(figsize=(13, 5))
rule_summary.plot(kind='bar', x='rule_name', y='mean_macro_f1', legend=False, ax=plt.gca())
plt.ylabel('Mean Macro F1')
plt.title('Mean Macro F1 by Rule')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'mean_macro_f1_by_rule.png', dpi=200)
plt.show()


## 10. Inspect One Best Pair

In [ ]:
pair_to_inspect = best_rule_per_pair.iloc[0]['pair_name']
inspect_df = all_pair_matrix_long[all_pair_matrix_long['pair_name'] == pair_to_inspect].copy()

inspect_cols = [
    'filename', 'class', 'y_true', 'reference_label',
    'cnn_prob_danger', 'dist_prob_danger',
    'cnn_pred', 'dist_pred',
    *RULE_PROB_COLS,
    *[col.replace('_prob', '_pred') for col in RULE_PROB_COLS],
    'rule_and_rule_label',
    'cascade_triggered', 'custom_uncertain_triggered',
    'near_boundary_rescue_triggered', 'near_boundary_rescue_used_distilbert',
    'or_rule_triggered', 'and_rule_unknown',
]
print(pair_to_inspect)
inspect_df[inspect_cols].head(50)


In [ ]:
UNKNOWN_PRED = -1

RULE_TO_PRED_CANDIDATES = {
    'cnn_only': ['rule_cnn_only_pred'],
    'distilbert_only': ['rule_distilbert_only_pred'],
    'override': ['rule_override_pred'],
    'weighted_average': ['rule_weighted_average_pred'],
    'cascade_correction': ['rule_cascade_correction_pred'],
    'probabilistic_late': ['rule_probabilistic_late_pred'],
    'custom_uncertain_handoff': ['rule_custom_uncertain_handoff_pred'],
    'cascade_near_boundary_rescue': ['rule_cascade_near_boundary_rescue_pred'],
    'or': ['rule_or_rule_pred', 'rule_or_pred'],
    'or_rule': ['rule_or_rule_pred', 'rule_or_pred'],
    'and': ['rule_and_rule_pred', 'rule_and_pred'],
    'and_rule': ['rule_and_rule_pred', 'rule_and_pred'],
    'ds_fusion': ['rule_ds_fusion_pred'],
    'ds_fusion_no_theta': ['rule_ds_fusion_no_theta_pred'],
}

def get_fusion_pred(pair_df, rule_name):
    rule_name = str(rule_name)

    for col in RULE_TO_PRED_CANDIDATES.get(rule_name, []):
        if col in pair_df.columns:
            return pair_df[col].astype(int)

    # Recompute missing columns if needed.
    if rule_name == 'probabilistic_late':
        if 'rule_probabilistic_late_prob' in pair_df.columns:
            return (pair_df['rule_probabilistic_late_prob'] >= FUSION_THRESHOLD).astype(int)

    if rule_name == 'ds_fusion_no_theta' and 'rule_ds_fusion_no_theta_prob' in pair_df.columns:
        return (pair_df['rule_ds_fusion_no_theta_prob'] >= FUSION_THRESHOLD).astype(int)
    if rule_name in ['or', 'or_rule']:
        return ((pair_df['cnn_pred'].astype(int) == 1) | (pair_df['dist_pred'].astype(int) == 1)).astype(int)

    if rule_name in ['and', 'and_rule']:
        cnn_pred = pair_df['cnn_pred'].astype(int)
        dist_pred = pair_df['dist_pred'].astype(int)
        return pd.Series(
            np.where(cnn_pred.eq(dist_pred), cnn_pred, UNKNOWN_PRED),
            index=pair_df.index,
        ).astype(int)

    raise KeyError(f'Could not find or recompute prediction column for rule: {rule_name}')


selection_rows = []

for _, row in all_metrics.iterrows():
    pair_name = row['pair_name']
    rule_name = row['rule_name']

    pair_df = all_pair_matrix_long[all_pair_matrix_long['pair_name'] == pair_name].copy()

    try:
        fusion_pred = get_fusion_pred(pair_df, rule_name)
    except KeyError as e:
        print(e)
        continue

    y_true = pair_df['y_true'].astype(int)
    cnn_pred = pair_df['cnn_pred'].astype(int)

    fusion_known = fusion_pred.isin([0, 1])

    cnn_wrong = cnn_pred.ne(y_true)
    cnn_correct = cnn_pred.eq(y_true)

    danger_samples = y_true.eq(1)
    cnn_wrong_danger = danger_samples & cnn_wrong

    fusion_correct = fusion_pred.eq(y_true) & fusion_known
    fusion_wrong = fusion_pred.ne(y_true) & fusion_known
    fusion_unknown = ~fusion_known

    corrected_cnn_danger_errors = int((cnn_wrong_danger & fusion_correct).sum())
    uncorrected_cnn_danger_errors = int((cnn_wrong_danger & ~fusion_correct).sum())
    unknown_cnn_danger_errors = int((cnn_wrong_danger & fusion_unknown).sum())

    corrected_cnn_errors = int((cnn_wrong & fusion_correct).sum())
    uncorrected_cnn_errors = int((cnn_wrong & ~fusion_correct).sum())
    unknown_cnn_errors = int((cnn_wrong & fusion_unknown).sum())

    cnn_correct_hurt_by_fusion = int((cnn_correct & fusion_wrong).sum())
    cnn_correct_unknown_by_fusion = int((cnn_correct & fusion_unknown).sum())

    net_gain_vs_cnn = corrected_cnn_errors - cnn_correct_hurt_by_fusion
    danger_net_gain_vs_cnn = corrected_cnn_danger_errors - cnn_correct_hurt_by_fusion

    out = row.to_dict()
    out.update({
        'corrected_cnn_danger_errors': corrected_cnn_danger_errors,
        'uncorrected_cnn_danger_errors': uncorrected_cnn_danger_errors,
        'unknown_cnn_danger_errors': unknown_cnn_danger_errors,
        'corrected_cnn_errors': corrected_cnn_errors,
        'uncorrected_cnn_errors': uncorrected_cnn_errors,
        'unknown_cnn_errors': unknown_cnn_errors,
        'cnn_correct_hurt_by_fusion': cnn_correct_hurt_by_fusion,
        'cnn_correct_unknown_by_fusion': cnn_correct_unknown_by_fusion,
        'net_gain_vs_cnn': net_gain_vs_cnn,
        'danger_net_gain_vs_cnn': danger_net_gain_vs_cnn,
    })
    selection_rows.append(out)

model_selection_ranked = pd.DataFrame(selection_rows)

model_selection_ranked = model_selection_ranked.sort_values(
    [
        'corrected_cnn_danger_errors',
        'net_gain_vs_cnn',
        'cnn_correct_hurt_by_fusion',
        'macro_f1',
        'f1_danger',
        'recall_danger',
        'accuracy',
    ],
    ascending=[False, False, True, False, False, False, False],
)

model_selection_ranked.to_csv(
    OUTPUT_DIR / 'model_selection_ranked_by_danger_rescue_then_f1_ALL_RULES.csv',
    index=False,
)

print('Rows exported:', len(model_selection_ranked))
print('Rules exported:')
print(model_selection_ranked['rule_name'].value_counts())

display_cols = [
    'cnn_model',
    'distilbert_model',
    'rule_name',
    'corrected_cnn_danger_errors',
    'uncorrected_cnn_danger_errors',
    'unknown_cnn_danger_errors',
    'corrected_cnn_errors',
    'cnn_correct_hurt_by_fusion',
    'net_gain_vs_cnn',
    'danger_net_gain_vs_cnn',
    'macro_f1',
    'f1_danger',
    'recall_danger',
    'precision_danger',
    'accuracy',
    'false_positive_rate',
    'false_negative_rate',
    'unknown_rows',
    'coverage_rate',
]

model_selection_ranked[display_cols].head(30)

In [ ]:
model_selection_ranked['rule_name'].value_counts()

In [ ]:
# Rank all CNN x DistilBERT x fusion-rule results by danger recall

ranked_by_danger_recall = all_metrics.copy()

# Optional: keep everything, including override.
# Usually exclude only and/and_rule if you do not want unknown-coverage rows.
# Comment this block out if you also want and_rule included.
ranked_by_danger_recall = ranked_by_danger_recall[
    ~ranked_by_danger_recall["rule_name"].isin(["and", "and_rule"])
].copy()

ranked_by_danger_recall = ranked_by_danger_recall.sort_values(
    [
        "recall_danger",
        "f1_danger",
        "macro_f1",
        "precision_danger",
        "accuracy",
    ],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)

display_cols = [
    "cnn_model",
    "distilbert_model",
    "pair_name",
    "rule_name",
    "recall_danger",
    "f1_danger",
    "macro_f1",
    "precision_danger",
    "accuracy",
    "false_positive_rate",
    "false_negative_rate",
    "unknown_rows",
    "coverage_rate",
    "n_rows",
]

display_cols = [col for col in display_cols if col in ranked_by_danger_recall.columns]

ranked_by_danger_recall_export = ranked_by_danger_recall[display_cols]

output_path = OUTPUT_DIR / "all_model_rule_results_ranked_by_danger_recall.csv"
ranked_by_danger_recall_export.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(ranked_by_danger_recall_export))
ranked_by_danger_recall_export.head(50)